# Repeated-Seed Base Models and Out-of-Fold Probability Generation

This notebook documents the redesigned repeated-seed base-model pipeline. For each planned seed, it trains group-aware five-fold SVM and IndoBERT models, creates out-of-fold probabilities for stacking, and produces label-blind locked-test base-model probabilities for later analysis.


## Release Boundary

This restricted code-inspection copy retains the analytical logic but does not include Banjarbaru review data, row-level audit records, predictions, checkpoints, or intermediate artifacts. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure the repeated-seed protocol

Imports the SVM, transformer, persistence, and runtime libraries. It defines the restricted project-root placeholder, planned seeds, five-fold outer and inner settings, model configurations, meta-feature names, and device policy.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import gc
import hashlib
import inspect
import json
import os
import platform
import random
import shutil
import sys
import time
import traceback
import uuid

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
import transformers
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed as transformers_set_seed,
)
from transformers.trainer_utils import get_last_checkpoint


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
ARTIFACT_ROOT = ROUND1_DIR / 'artifacts' / 'r1_06_repeated_seed_base_oof_v2'

TRAIN_PATH = PROJECT_ROOT / 'data' / 'splits' / 'train.csv'
VALIDATION_PATH = PROJECT_ROOT / 'data' / 'splits' / 'validation.csv'
LOCKED_TEST_PATH = PROJECT_ROOT / 'data' / 'final_test_locked' / 'test_locked.csv'
SPLIT_METADATA_PATH = PROJECT_ROOT / 'data' / 'splits' / 'split_metadata.json'

PLANNED_SEEDS = [42, 52, 62]
RUN_SEEDS = [62]
MAX_NEW_FOLDS_PER_RUN: int | None = None  
KEEP_FINAL_FOLD_MODELS = True
PRUNE_COMPLETED_INDO_BERT_CHECKPOINTS = True

LABEL_NAMES = ['negatif', 'netral', 'positif']
LABEL_TO_ID = {label: index for index, label in enumerate(LABEL_NAMES)}
N_CLASSES = len(LABEL_NAMES)
GROUP_COLUMN = 'Actual Place Name'
LABEL_COLUMN = 'sentiment_label'

OUTER_N_SPLITS = 5
INNER_N_SPLITS = 5
PROTOCOL_VERSION = 'r1_06_protocol_v2_outer_train_refit'

SVM_CONFIG = {
    'text_column': 'text_lower',
    'ngram_range': [1, 3],
    'min_df': 2,
    'max_df': 0.95,
    'max_features': 10_000,
    'sublinear_tf': True,
    'lowercase': False,
    'strip_accents': 'unicode',
    'kernel': 'linear',
    'C': 1.0,
    'class_weight': 'balanced',
    'probability': True,
}

BERT_CONFIG = {
    'model_name': 'indobenchmark/indobert-base-p1',
    'text_column': 'text_clean',
    'max_length': 128,
    'learning_rate': 2e-5,
    'train_batch_size': 16,
    'eval_batch_size': 32,
    'num_train_epochs': 4,
    'warmup_ratio': 0.10,
    'weight_decay': 0.01,
    'gradient_accumulation_steps': 1,
    'max_grad_norm': 1.0,
    'early_stopping_patience': 2,
    'checkpoint_metric': 'macro_f1',
    'save_total_limit_during_training': 2,
    'outer_train_refit_after_selection': True,
    'refit_seed_purpose': 9,
}

META_FEATURE_NAMES = [
    'svm_prob_negatif', 'svm_prob_netral', 'svm_prob_positif',
    'bert_prob_negatif', 'bert_prob_netral', 'bert_prob_positif',
    'svm_confidence', 'bert_confidence',
    'svm_entropy', 'bert_entropy',
    'pred_disagreement', 'prob_disagreement',
]

CUDA_AVAILABLE = torch.cuda.is_available()
if not CUDA_AVAILABLE:
    print('WARNING: CUDA tidak tersedia. R1_06 berjalan di CPU dengan fp16=False; IndoBERT per fold dapat memakan waktu sangat lama.')

DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')
USE_FP16 = DEVICE.type == 'cuda'

print(f'Restricted project root placeholder: {PROJECT_ROOT}')
print(f'Artifact root: {ARTIFACT_ROOT}')
print(f'Run seeds: {RUN_SEEDS} of planned {PLANNED_SEEDS}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')
print(f'PyTorch: {torch.__version__} | Transformers: {transformers.__version__}')


### Load restricted partitions and lock the analysis contract

Reads authorized train and validation labels, reads locked-test text without its label column, validates schemas and split metadata, maps the silver labels, and creates hashes for the inputs and scientific protocol.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_hash(value: dict[str, Any]) -> str:
    encoded = json.dumps(value, ensure_ascii=True, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


required_train_columns = {GROUP_COLUMN, LABEL_COLUMN, SVM_CONFIG['text_column'], BERT_CONFIG['text_column']}
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
locked_test_text_df = pd.read_csv(
    LOCKED_TEST_PATH, usecols=list({SVM_CONFIG['text_column'], BERT_CONFIG['text_column']})
)

for split_name, frame in {'train': train_df, 'validation': validation_df}.items():
    missing = sorted(required_train_columns.difference(frame.columns))
    if missing:
        raise ValueError(f'{split_name} kehilangan kolom wajib: {missing}')

for text_column in {SVM_CONFIG['text_column'], BERT_CONFIG['text_column']}:
    if text_column not in locked_test_text_df.columns:
        raise ValueError(f'Locked test kehilangan {text_column} untuk inference.')

with SPLIT_METADATA_PATH.open(encoding='utf-8') as handle:
    split_metadata = json.load(handle)

expected_n = {
    'train': split_metadata['splits']['train']['n_samples'],
    'validation': split_metadata['splits']['validation']['n_samples'],
    'locked_test': split_metadata['splits']['test']['n_samples'],
}
assert len(train_df) == expected_n['train'], (len(train_df), expected_n['train'])
assert len(validation_df) == expected_n['validation'], (len(validation_df), expected_n['validation'])
assert len(locked_test_text_df) == expected_n['locked_test'], (len(locked_test_text_df), expected_n['locked_test'])
assert split_metadata['group_column'] == GROUP_COLUMN
assert split_metadata['label_column'] == LABEL_COLUMN
assert split_metadata['leakage_check']['val_test_overlap'] == 0

def map_labels(series: pd.Series, split_name: str) -> np.ndarray:
    normalized = series.astype(str).str.strip().str.lower()
    unknown = sorted(set(normalized).difference(LABEL_TO_ID))
    if unknown:
        raise ValueError(f'{split_name} mempunyai label tidak dikenal: {unknown}')
    return normalized.map(LABEL_TO_ID).to_numpy(dtype=np.int64)


train_y = map_labels(train_df[LABEL_COLUMN], 'train')
validation_y = map_labels(validation_df[LABEL_COLUMN], 'validation')
train_groups = train_df[GROUP_COLUMN].fillna('').astype(str).str.strip().to_numpy()
if (train_groups == '').any():
    raise ValueError('Actual Place Name kosong ditemukan pada train.')

TRAIN_ROW_IDS = np.arange(len(train_df), dtype=np.int64)
VALIDATION_ROW_IDS = np.arange(len(validation_df), dtype=np.int64)
LOCKED_TEST_ROW_IDS = np.arange(len(locked_test_text_df), dtype=np.int64)
LOCKED_TEST_SVM_TEXTS = locked_test_text_df[SVM_CONFIG['text_column']].fillna('').astype(str).tolist()
LOCKED_TEST_BERT_TEXTS = locked_test_text_df[BERT_CONFIG['text_column']].fillna('').astype(str).tolist()

input_hashes = {
    'train_csv_sha256': sha256_file(TRAIN_PATH),
    'validation_csv_sha256': sha256_file(VALIDATION_PATH),
    'locked_test_csv_sha256': sha256_file(LOCKED_TEST_PATH),
    'split_metadata_sha256': sha256_file(SPLIT_METADATA_PATH),
}
scientific_protocol = {
    'protocol_version': PROTOCOL_VERSION,
    'label_names': LABEL_NAMES,
    'group_column': GROUP_COLUMN,
    'outer_n_splits': OUTER_N_SPLITS,
    'inner_n_splits': INNER_N_SPLITS,
    'svm_config': SVM_CONFIG,
    'bert_config': BERT_CONFIG,
    'meta_feature_names': META_FEATURE_NAMES,
    'prediction_generator': 'five_fold_outer_ensemble_for_validation_and_locked_test',
    'locked_test_policy': 'label_blind_prediction_only_no_metrics_in_r1_06',
}
PROTOCOL_HASH = canonical_json_hash(scientific_protocol)

print(f'Train: {len(train_df):,} | Validation: {len(validation_df):,} | Locked test prediction rows: {len(locked_test_text_df):,}')
print('Train class counts:', dict(zip(LABEL_NAMES, np.bincount(train_y, minlength=N_CLASSES))))
print('Validation class counts:', dict(zip(LABEL_NAMES, np.bincount(validation_y, minlength=N_CLASSES))))
print(f'Protocol hash: {PROTOCOL_HASH}')


### Define resumable artifact and protocol utilities

Defines atomic writers, JSON readers, group digests, memory cleanup, and protocol-manifest checks. In the authorized environment these utilities prevent mixing results from incompatible inputs or configurations.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    with temp_path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, default=str)
    os.replace(temp_path, path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    frame.to_csv(temp_path, index=False)
    os.replace(temp_path, path)


def atomic_joblib_dump(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    joblib.dump(value, temp_path)
    os.replace(temp_path, path)


def read_json(path: Path) -> dict[str, Any]:
    with path.open(encoding='utf-8') as handle:
        return json.load(handle)


def group_digest(value: str) -> str:
    return hashlib.sha256(f'r1_06_group::{value}'.encode('utf-8')).hexdigest()


def clear_accelerator_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


protocol_path = ARTIFACT_ROOT / 'protocol_v2.json'
protocol_payload = {
    'created_at_utc': utc_now(),
    'scientific_protocol': scientific_protocol,
    'protocol_hash': PROTOCOL_HASH,
    'input_hashes': input_hashes,
    'planned_seeds': PLANNED_SEEDS,
    'environment_at_creation': {
        'python': sys.version, 'platform': platform.platform(), 'numpy': np.__version__,
        'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
        'torch': torch.__version__, 'transformers': transformers.__version__,
    },
}
if protocol_path.exists():
    existing_protocol = read_json(protocol_path)
    if existing_protocol.get('protocol_hash') != PROTOCOL_HASH:
        raise RuntimeError('Protocol hash berbeda dari artifact lama. Jangan mencampur rerun dengan recipe yang berbeda.')
    if existing_protocol.get('input_hashes') != input_hashes:
        raise RuntimeError('Input hash berubah dari artifact lama. Periksa data frozen sebelum melanjutkan.')
else:
    atomic_write_json(protocol_path, protocol_payload)

print(f'Protocol manifest ready: {protocol_path}')


### Define common modelling and probability utilities

Defines deterministic seeding, the transformer dataset and weighted trainer, probability prediction and validation routines, entropy calculations, metric summaries, and standard prediction-frame layouts.


In [ ]:
def derive_seed(base_seed: int, outer_fold: int = 0, purpose: int = 0) -> int:
    return int((base_seed * 10_000 + outer_fold * 101 + purpose * 7) % (2**31 - 1))


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    transformers_set_seed(seed)


class TextClassificationDataset(Dataset):
    def __init__(self, texts: list[str], tokenizer: Any, max_length: int, labels: np.ndarray | None = None):
        self.texts = [str(text) for text in texts]
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = labels

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        encoded = self.tokenizer(
            self.texts[index], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        item = {key: value.squeeze(0) for key, value in encoded.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return item


class WeightedTrainer(Trainer):
    def __init__(self, class_weights: np.ndarray, *args: Any, **kwargs: Any):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model: Any, inputs: dict[str, Any], return_outputs: bool = False, num_items_in_batch: Any = None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_trainer_metrics(eval_prediction: Any) -> dict[str, float]:
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=1)
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(labels, predictions, average='macro', zero_division=0)),
    }


def predict_probabilities_torch(model: Any, texts: list[str], tokenizer: Any) -> np.ndarray:
    dataset = TextClassificationDataset(texts, tokenizer, BERT_CONFIG['max_length'], labels=None)
    loader = DataLoader(dataset, batch_size=BERT_CONFIG['eval_batch_size'], shuffle=False, num_workers=0)
    model.eval()
    batches: list[np.ndarray] = []
    with torch.inference_mode():
        for batch in loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            logits = model(**batch).logits
            batches.append(torch.softmax(logits, dim=1).detach().cpu().numpy())
    probabilities = np.concatenate(batches, axis=0)
    validate_probability_matrix(probabilities, 'IndoBERT prediction')
    return probabilities


def validate_probability_matrix(probabilities: np.ndarray, name: str, tolerance: float = 1e-6) -> None:
    if probabilities.ndim != 2 or probabilities.shape[1] != N_CLASSES:
        raise ValueError(f'{name}: shape probability salah: {probabilities.shape}')
    if not np.isfinite(probabilities).all():
        raise ValueError(f'{name}: NaN atau infinity ditemukan.')
    if (probabilities < -tolerance).any() or (probabilities > 1 + tolerance).any():
        raise ValueError(f'{name}: probability di luar rentang [0, 1].')
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=tolerance):
        raise ValueError(f'{name}: jumlah probability kelas tidak sama dengan 1.')


def align_svc_probabilities(probabilities: np.ndarray, learned_classes: np.ndarray) -> np.ndarray:
    aligned = np.zeros((len(probabilities), N_CLASSES), dtype=np.float64)
    for source_column, class_id in enumerate(learned_classes):
        aligned[:, int(class_id)] = probabilities[:, source_column]
    validate_probability_matrix(aligned, 'SVM prediction')
    return aligned


def probability_entropy(probabilities: np.ndarray) -> np.ndarray:
    clipped = np.clip(probabilities, 1e-12, 1.0)
    return -(clipped * np.log(clipped)).sum(axis=1) / np.log(N_CLASSES)


def metric_summary(y_true: np.ndarray, probabilities: np.ndarray) -> dict[str, float]:
    predictions = np.argmax(probabilities, axis=1)
    return {
        'n': int(len(y_true)),
        'accuracy': float(accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro', zero_division=0)),
    }


def prediction_frame(row_ids: np.ndarray, probabilities: np.ndarray, prefix: str, outer_fold: int | None = None, y_silver: np.ndarray | None = None) -> pd.DataFrame:
    validate_probability_matrix(probabilities, f'{prefix} frame')
    frame = pd.DataFrame({'row_id': row_ids.astype(np.int64)})
    if outer_fold is not None:
        frame['outer_fold'] = int(outer_fold)
    if y_silver is not None:
        frame['y_silver'] = y_silver.astype(np.int64)
    for class_index, class_name in enumerate(LABEL_NAMES):
        frame[f'{prefix}_prob_{class_name}'] = probabilities[:, class_index]
    frame[f'{prefix}_prediction'] = np.argmax(probabilities, axis=1).astype(np.int64)
    frame[f'{prefix}_confidence'] = probabilities.max(axis=1)
    frame[f'{prefix}_entropy'] = probability_entropy(probabilities)
    return frame


def model_probability_columns(prefix: str) -> list[str]:
    return [f'{prefix}_prob_{class_name}' for class_name in LABEL_NAMES]


### Build group-aware fold plans and resumable seed state

Creates five outer Actual Place Name group-aware folds, inner folds for IndoBERT epoch selection, state tracking for interrupted runs, and private fold-assignment records.


In [ ]:
@dataclass
class FoldPlan:
    outer_fold: int
    outer_train_idx: np.ndarray
    outer_val_idx: np.ndarray
    inner_train_idx: np.ndarray
    inner_eval_idx: np.ndarray
    svm_seed: int
    bert_seed: int
    inner_seed: int


def assert_group_disjoint(left_idx: np.ndarray, right_idx: np.ndarray, description: str) -> None:
    overlap = set(train_groups[left_idx]).intersection(train_groups[right_idx])
    if overlap:
        raise RuntimeError(f'{description}: ditemukan {len(overlap)} group overlap.')


def make_seed_plan(base_seed: int) -> list[FoldPlan]:
    outer_splitter = StratifiedGroupKFold(
        n_splits=OUTER_N_SPLITS, shuffle=True, random_state=base_seed,
    )
    plans: list[FoldPlan] = []
    coverage = np.zeros(len(train_df), dtype=np.int64)

    for outer_fold, (outer_train_idx, outer_val_idx) in enumerate(
        outer_splitter.split(train_df, train_y, groups=train_groups), start=1
    ):
        assert_group_disjoint(outer_train_idx, outer_val_idx, f'outer fold {outer_fold}')
        inner_seed = derive_seed(base_seed, outer_fold, purpose=2)
        inner_splitter = StratifiedGroupKFold(
            n_splits=INNER_N_SPLITS, shuffle=True, random_state=inner_seed,
        )
        inner_train_relative, inner_eval_relative = next(
            inner_splitter.split(
                train_df.iloc[outer_train_idx], train_y[outer_train_idx], groups=train_groups[outer_train_idx]
            )
        )
        inner_train_idx = outer_train_idx[inner_train_relative]
        inner_eval_idx = outer_train_idx[inner_eval_relative]
        assert_group_disjoint(inner_train_idx, inner_eval_idx, f'inner fold {outer_fold}')
        assert_group_disjoint(inner_train_idx, outer_val_idx, f'inner-train versus outer-val fold {outer_fold}')
        assert_group_disjoint(inner_eval_idx, outer_val_idx, f'inner-eval versus outer-val fold {outer_fold}')
        coverage[outer_val_idx] += 1
        plans.append(FoldPlan(
            outer_fold=outer_fold,
            outer_train_idx=outer_train_idx, outer_val_idx=outer_val_idx,
            inner_train_idx=inner_train_idx, inner_eval_idx=inner_eval_idx,
            svm_seed=derive_seed(base_seed, outer_fold, purpose=1),
            bert_seed=derive_seed(base_seed, outer_fold, purpose=3),
            inner_seed=inner_seed,
        ))

    if not np.all(coverage == 1):
        raise RuntimeError('OOF coverage tidak tepat: setiap train row harus muncul tepat sekali sebagai outer validation.')
    return plans


def initial_seed_state(seed: int) -> dict[str, Any]:
    return {
        'seed': seed, 'protocol_hash': PROTOCOL_HASH, 'input_hashes': input_hashes,
        'status': 'pending', 'created_at_utc': utc_now(), 'updated_at_utc': utc_now(),
        'folds': {str(fold): {'status': 'pending'} for fold in range(1, OUTER_N_SPLITS + 1)},
    }


def seed_directory(seed: int) -> Path:
    return ARTIFACT_ROOT / f'seed_{seed}'


def seed_state_path(seed_dir: Path) -> Path:
    return seed_dir / 'run_state.json'


def load_or_create_seed_state(seed: int) -> tuple[Path, dict[str, Any]]:
    seed_dir = seed_directory(seed)
    seed_dir.mkdir(parents=True, exist_ok=True)
    path = seed_state_path(seed_dir)
    if path.exists():
        state = read_json(path)
        if state.get('protocol_hash') != PROTOCOL_HASH or state.get('input_hashes') != input_hashes:
            raise RuntimeError(f'Seed {seed} mempunyai protocol atau input hash berbeda. Jangan resume artifact yang tidak kompatibel.')
    else:
        state = initial_seed_state(seed)
        atomic_write_json(path, state)
    return seed_dir, state


def update_seed_state(seed_dir: Path, state: dict[str, Any]) -> None:
    state['updated_at_utc'] = utc_now()
    atomic_write_json(seed_state_path(seed_dir), state)


def mark_fold_state(seed_dir: Path, fold: int, status: str, **details: Any) -> dict[str, Any]:
    state = read_json(seed_state_path(seed_dir))
    state['status'] = 'running' if status not in {'seed_completed', 'seed_failed'} else status
    fold_state = state['folds'][str(fold)]
    fold_state.update({'status': status, 'updated_at_utc': utc_now(), **details})
    update_seed_state(seed_dir, state)
    return state


def persist_seed_plan(seed_dir: Path, plans: list[FoldPlan]) -> None:
    outer_rows = []
    inner_rows = []
    summaries = []
    for plan in plans:
        for row_id in plan.outer_val_idx:
            outer_rows.append({
                'row_id': int(row_id), 'outer_fold': plan.outer_fold,
                'group_digest': group_digest(train_groups[row_id]), 'y_silver': int(train_y[row_id]),
            })
        for partition, indices in [('inner_train', plan.inner_train_idx), ('inner_eval', plan.inner_eval_idx)]:
            for row_id in indices:
                inner_rows.append({
                    'outer_fold': plan.outer_fold, 'row_id': int(row_id), 'partition': partition,
                    'group_digest': group_digest(train_groups[row_id]), 'y_silver': int(train_y[row_id]),
                })
        summaries.append({
            'outer_fold': plan.outer_fold, 'outer_train_n': int(len(plan.outer_train_idx)),
            'outer_val_n': int(len(plan.outer_val_idx)), 'inner_train_n': int(len(plan.inner_train_idx)),
            'inner_eval_n': int(len(plan.inner_eval_idx)), 'svm_seed': plan.svm_seed,
            'bert_seed': plan.bert_seed, 'inner_seed': plan.inner_seed,
        })
    atomic_write_csv(seed_dir / 'train_outer_fold_assignments.csv', pd.DataFrame(outer_rows).sort_values('row_id'))
    atomic_write_csv(seed_dir / 'indobert_inner_split_assignments.csv', pd.DataFrame(inner_rows).sort_values(['outer_fold', 'row_id']))
    atomic_write_json(seed_dir / 'fold_structure.json', {'seed': plans[0].svm_seed // 10_000, 'folds': summaries})


def fold_directory(seed_dir: Path, fold: int) -> Path:
    directory = seed_dir / f'fold_{fold}'
    directory.mkdir(parents=True, exist_ok=True)
    return directory


print('Fold/state helpers ready.')


### Fit the TF-IDF SVM within each outer fold

Fits the configured balanced linear SVM on outer-training rows, then writes private OOF, validation, and label-blind locked-test probability files and optional model exports for that fold.


In [ ]:
def run_svm_fold(plan: FoldPlan, fold_dir: Path) -> dict[str, Any]:
    model_dir = fold_dir / 'models' / 'svm'
    prediction_dir = fold_dir / 'predictions'
    model_dir.mkdir(parents=True, exist_ok=True)
    prediction_dir.mkdir(parents=True, exist_ok=True)

    seed_everything(plan.svm_seed)
    vectorizer = TfidfVectorizer(
        ngram_range=tuple(SVM_CONFIG['ngram_range']), min_df=SVM_CONFIG['min_df'],
        max_df=SVM_CONFIG['max_df'], max_features=SVM_CONFIG['max_features'],
        sublinear_tf=SVM_CONFIG['sublinear_tf'], lowercase=SVM_CONFIG['lowercase'],
        strip_accents=SVM_CONFIG['strip_accents'],
    )
    train_texts = train_df.iloc[plan.outer_train_idx][SVM_CONFIG['text_column']].fillna('').astype(str).tolist()
    oof_texts = train_df.iloc[plan.outer_val_idx][SVM_CONFIG['text_column']].fillna('').astype(str).tolist()
    validation_texts = validation_df[SVM_CONFIG['text_column']].fillna('').astype(str).tolist()

    matrix_train = vectorizer.fit_transform(train_texts)
    model = SVC(
        kernel=SVM_CONFIG['kernel'], C=SVM_CONFIG['C'], class_weight=SVM_CONFIG['class_weight'],
        probability=True, random_state=plan.svm_seed,
    )
    started = time.time()
    model.fit(matrix_train, train_y[plan.outer_train_idx])
    elapsed_seconds = time.time() - started

    def svm_proba(texts: list[str]) -> np.ndarray:
        matrix = vectorizer.transform(texts)
        return align_svc_probabilities(model.predict_proba(matrix), model.classes_)

    oof_proba = svm_proba(oof_texts)
    validation_proba = svm_proba(validation_texts)
    locked_test_proba = svm_proba(LOCKED_TEST_SVM_TEXTS)

    atomic_write_csv(
        prediction_dir / 'svm_oof.csv',
        prediction_frame(TRAIN_ROW_IDS[plan.outer_val_idx], oof_proba, 'svm', plan.outer_fold, train_y[plan.outer_val_idx]),
    )
    atomic_write_csv(
        prediction_dir / 'svm_validation.csv',
        prediction_frame(VALIDATION_ROW_IDS, validation_proba, 'svm', y_silver=validation_y),
    )
    atomic_write_csv(
        prediction_dir / 'svm_locked_test_blind.csv',
        prediction_frame(LOCKED_TEST_ROW_IDS, locked_test_proba, 'svm'),
    )

    if KEEP_FINAL_FOLD_MODELS:
        atomic_joblib_dump(model_dir / 'model.joblib', model)
        atomic_joblib_dump(model_dir / 'vectorizer.joblib', vectorizer)

    result = {
        'model': 'SVM', 'fold': plan.outer_fold, 'training_seconds': elapsed_seconds,
        'oof_metrics': metric_summary(train_y[plan.outer_val_idx], oof_proba),
        'class_order': LABEL_NAMES, 'n_tfidf_features': int(matrix_train.shape[1]),
    }
    atomic_write_json(fold_dir / 'svm_fold_summary.json', result)
    return result


### Select and refit IndoBERT within each outer fold

Uses the inner group-aware split for epoch selection, then refits IndoBERT on the complete outer-training partition before exporting the model and its probability predictions. Intermediate checkpoints are pruned only after final fold artifacts are complete.


In [ ]:
def model_export_ready(model_dir: Path) -> bool:
    marker = model_dir / 'export_complete.json'
    has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'pytorch_model.bin').exists()
    if not (marker.exists() and (model_dir / 'config.json').exists() and has_weights):
        return False
    try:
        return read_json(marker).get('training_stage') == 'outer_train_refit'
    except Exception:
        return False


class FoldCheckpointCallback(TrainerCallback):
    def __init__(self, seed_dir: Path, fold: int, stage: str):
        self.seed_dir = seed_dir
        self.fold = fold
        self.stage = stage

    def on_save(self, args: Any, state: Any, control: Any, **kwargs: Any):
        checkpoint = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        mark_fold_state(
            self.seed_dir, self.fold, 'checkpointed', training_stage=self.stage,
            last_checkpoint=str(checkpoint), global_step=int(state.global_step), epoch=float(state.epoch or 0),
        )
        return control


def build_training_arguments(training_dir: Path, fold_seed: int, num_epochs: int, stage: str) -> TrainingArguments:
    if stage not in {'selection', 'refit'}:
        raise ValueError(f'Tahap IndoBERT tidak dikenal: {stage}')
    is_selection = stage == 'selection'
    kwargs: dict[str, Any] = {
        'output_dir': str(training_dir),
        'num_train_epochs': int(num_epochs),
        'per_device_train_batch_size': BERT_CONFIG['train_batch_size'],
        'per_device_eval_batch_size': BERT_CONFIG['eval_batch_size'],
        'gradient_accumulation_steps': BERT_CONFIG['gradient_accumulation_steps'],
        'learning_rate': BERT_CONFIG['learning_rate'],
        'warmup_ratio': BERT_CONFIG['warmup_ratio'],
        'weight_decay': BERT_CONFIG['weight_decay'],
        'max_grad_norm': BERT_CONFIG['max_grad_norm'],
        'save_strategy': 'epoch',
        'save_total_limit': BERT_CONFIG['save_total_limit_during_training'] if is_selection else 1,
        'logging_strategy': 'steps',
        'logging_steps': 25,
        'report_to': 'none',
        'seed': fold_seed,
        'fp16': USE_FP16,
        'dataloader_num_workers': 0,
    }
    parameter_names = inspect.signature(TrainingArguments.__init__).parameters
    if is_selection:
        kwargs.update({
            'load_best_model_at_end': True,
            'metric_for_best_model': BERT_CONFIG['checkpoint_metric'],
            'greater_is_better': True,
        })
        if 'eval_strategy' in parameter_names:
            kwargs['eval_strategy'] = 'epoch'
        else:
            kwargs['evaluation_strategy'] = 'epoch'
    else:
        kwargs['load_best_model_at_end'] = False
        if 'eval_strategy' in parameter_names:
            kwargs['eval_strategy'] = 'no'
        else:
            kwargs['evaluation_strategy'] = 'no'
    if 'data_seed' in parameter_names:
        kwargs['data_seed'] = fold_seed
    return TrainingArguments(**kwargs)


def calculate_class_weights(labels: np.ndarray) -> np.ndarray:
    counts = np.bincount(labels, minlength=N_CLASSES).astype(np.float64)
    if (counts == 0).any():
        raise ValueError(f'IndoBERT training kehilangan kelas: {counts.tolist()}')
    return len(labels) / (N_CLASSES * counts)


def load_exported_bert(model_dir: Path) -> tuple[Any, Any]:
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    return model, tokenizer


def selected_epoch_from_checkpoint(best_checkpoint: str | None, fallback_epoch: float | None) -> int:
    if not best_checkpoint:
        raise RuntimeError('Inner selection tidak menghasilkan best checkpoint.')
    checkpoint_state = Path(best_checkpoint) / 'trainer_state.json'
    selected_epoch_value = fallback_epoch
    if checkpoint_state.exists():
        selected_epoch_value = read_json(checkpoint_state).get('epoch', selected_epoch_value)
    if selected_epoch_value is None:
        raise RuntimeError('Epoch checkpoint terbaik tidak dapat ditentukan.')
    return max(1, min(int(BERT_CONFIG['num_train_epochs']), int(np.ceil(float(selected_epoch_value)))))


def prune_checkpoint_directory(path: Path, description: str) -> bool:
    if not PRUNE_COMPLETED_INDO_BERT_CHECKPOINTS or not path.exists():
        return False
    try:
        shutil.rmtree(path)
        print(f'{description}: checkpoint intermediate dibersihkan setelah artifact final aman.')
        return True
    except OSError as error:
        print(f'{description}: checkpoint tidak dapat dibersihkan ({error!r}). Artifact final tetap valid.')
        return False


def load_or_run_epoch_selection(plan: FoldPlan, seed_dir: Path, fold_dir: Path) -> dict[str, Any]:
    selection_summary_path = fold_dir / 'indobert_epoch_selection.json'
    if selection_summary_path.exists():
        selection = read_json(selection_summary_path)
        if selection.get('selection_seed') != plan.bert_seed:
            raise RuntimeError(f'Fold {plan.outer_fold}: selection seed tidak kompatibel.')
        selected_epoch = int(selection.get('selected_epoch', 0))
        if not 1 <= selected_epoch <= int(BERT_CONFIG['num_train_epochs']):
            raise RuntimeError(f'Fold {plan.outer_fold}: selected_epoch tidak valid.')
        selection['loaded_from_completed_selection'] = True
        return selection

    selection_dir = fold_dir / 'selection_checkpoints'
    selection_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(plan.bert_seed)
    tokenizer = AutoTokenizer.from_pretrained(BERT_CONFIG['model_name'])
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_CONFIG['model_name'], num_labels=N_CLASSES,
        id2label={index: label for index, label in enumerate(LABEL_NAMES)}, label2id=LABEL_TO_ID,
    ).to(DEVICE)
    inner_train_y = train_y[plan.inner_train_idx]
    inner_eval_y = train_y[plan.inner_eval_idx]
    inner_train_texts = train_df.iloc[plan.inner_train_idx][BERT_CONFIG['text_column']].fillna('').astype(str).tolist()
    inner_eval_texts = train_df.iloc[plan.inner_eval_idx][BERT_CONFIG['text_column']].fillna('').astype(str).tolist()
    class_weights = calculate_class_weights(inner_train_y)
    trainer = WeightedTrainer(
        class_weights=class_weights, model=model,
        args=build_training_arguments(selection_dir, plan.bert_seed, BERT_CONFIG['num_train_epochs'], 'selection'),
        train_dataset=TextClassificationDataset(inner_train_texts, tokenizer, BERT_CONFIG['max_length'], inner_train_y),
        eval_dataset=TextClassificationDataset(inner_eval_texts, tokenizer, BERT_CONFIG['max_length'], inner_eval_y),
        compute_metrics=compute_trainer_metrics,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=BERT_CONFIG['early_stopping_patience']),
            FoldCheckpointCallback(seed_dir, plan.outer_fold, 'inner_epoch_selection'),
        ],
    )
    resume_checkpoint = get_last_checkpoint(str(selection_dir))
    started = time.time()
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint) if resume_checkpoint else trainer.train()
    selection = {
        'training_stage': 'inner_epoch_selection', 'fold': plan.outer_fold, 'selection_seed': plan.bert_seed,
        'inner_train_n': int(len(plan.inner_train_idx)), 'inner_eval_n': int(len(plan.inner_eval_idx)),
        'class_weights_inner_train': class_weights.tolist(),
        'best_metric': trainer.state.best_metric, 'best_model_checkpoint': trainer.state.best_model_checkpoint,
        'selected_epoch': selected_epoch_from_checkpoint(trainer.state.best_model_checkpoint, trainer.state.epoch),
        'final_selection_epoch': float(trainer.state.epoch or 0),
        'selection_training_seconds': time.time() - started,
        'selection_resume_checkpoint': resume_checkpoint,
        'selection_train_metrics': getattr(train_result, 'metrics', {}),
        'completed_at_utc': utc_now(),
    }
    atomic_write_json(selection_summary_path, selection)
    del trainer, model, tokenizer
    clear_accelerator_memory()
    selection['selection_checkpoints_pruned'] = prune_checkpoint_directory(selection_dir, f'Fold {plan.outer_fold} inner selection')
    atomic_write_json(selection_summary_path, selection)
    return selection


def train_or_load_bert_fold(plan: FoldPlan, seed_dir: Path, fold_dir: Path) -> tuple[Any, Any, dict[str, Any]]:
    model_dir = fold_dir / 'models' / 'indobert_outer_refit_model'
    model_dir.mkdir(parents=True, exist_ok=True)
    if model_export_ready(model_dir):
        model, tokenizer = load_exported_bert(model_dir)
        metadata = read_json(model_dir / 'export_complete.json')
        metadata['loaded_from_completed_export'] = True
        return model, tokenizer, metadata

    selection = load_or_run_epoch_selection(plan, seed_dir, fold_dir)
    selected_epoch = int(selection['selected_epoch'])
    refit_seed = derive_seed(plan.bert_seed, purpose=BERT_CONFIG['refit_seed_purpose'])
    refit_dir = fold_dir / 'refit_checkpoints'
    refit_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(refit_seed)
    tokenizer = AutoTokenizer.from_pretrained(BERT_CONFIG['model_name'])
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_CONFIG['model_name'], num_labels=N_CLASSES,
        id2label={index: label for index, label in enumerate(LABEL_NAMES)}, label2id=LABEL_TO_ID,
    ).to(DEVICE)
    outer_train_y = train_y[plan.outer_train_idx]
    outer_train_texts = train_df.iloc[plan.outer_train_idx][BERT_CONFIG['text_column']].fillna('').astype(str).tolist()
    outer_class_weights = calculate_class_weights(outer_train_y)
    trainer = WeightedTrainer(
        class_weights=outer_class_weights, model=model,
        args=build_training_arguments(refit_dir, refit_seed, selected_epoch, 'refit'),
        train_dataset=TextClassificationDataset(outer_train_texts, tokenizer, BERT_CONFIG['max_length'], outer_train_y),
        callbacks=[FoldCheckpointCallback(seed_dir, plan.outer_fold, 'outer_train_refit')],
    )
    resume_checkpoint = get_last_checkpoint(str(refit_dir))
    started = time.time()
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint) if resume_checkpoint else trainer.train()
    elapsed_seconds = time.time() - started
    trainer.save_model(str(model_dir))
    tokenizer.save_pretrained(str(model_dir))
    metadata = {
        'training_stage': 'outer_train_refit', 'model': 'IndoBERT', 'fold': plan.outer_fold,
        'base_model': BERT_CONFIG['model_name'], 'class_order': LABEL_NAMES,
        'outer_train_n': int(len(plan.outer_train_idx)), 'outer_train_class_weights': outer_class_weights.tolist(),
        'refit_seed': refit_seed, 'refit_epochs': selected_epoch,
        'refit_resume_checkpoint': resume_checkpoint, 'refit_training_seconds': elapsed_seconds,
        'refit_train_metrics': getattr(train_result, 'metrics', {}),
        'selection': selection, 'exported_at_utc': utc_now(),
    }
    atomic_write_json(model_dir / 'export_complete.json', metadata)
    metadata['refit_checkpoints_pruned'] = prune_checkpoint_directory(refit_dir, f'Fold {plan.outer_fold} outer-train refit')
    atomic_write_json(model_dir / 'export_complete.json', metadata)
    return trainer.model, tokenizer, metadata


### Run a fold with completion and privacy checks

Verifies required fold artifacts, resumes only incomplete work, generates SVM and IndoBERT probabilities, enforces the absence of locked-test labels in blind files, records checksums, and updates the per-seed state.


In [ ]:
def required_fold_paths(fold_dir: Path) -> list[Path]:
    prediction_dir = fold_dir / 'predictions'
    return [
        prediction_dir / 'svm_oof.csv', prediction_dir / 'svm_validation.csv', prediction_dir / 'svm_locked_test_blind.csv',
        prediction_dir / 'bert_oof.csv', prediction_dir / 'bert_validation.csv', prediction_dir / 'bert_locked_test_blind.csv',
        fold_dir / 'indobert_epoch_selection.json',
        fold_dir / 'models' / 'indobert_outer_refit_model' / 'export_complete.json',
        fold_dir / 'fold_summary.json',
    ]


def validate_prediction_file(path: Path, prefix: str, expected_ids: np.ndarray, expect_label: bool) -> None:
    frame = pd.read_csv(path)
    if len(frame) != len(expected_ids) or not np.array_equal(frame['row_id'].to_numpy(dtype=np.int64), expected_ids):
        raise ValueError(f'{path.name}: row id tidak cocok.')
    if expect_label and 'y_silver' not in frame.columns:
        raise ValueError(f'{path.name}: y_silver seharusnya tersedia.')
    if not expect_label and 'y_silver' in frame.columns:
        raise ValueError(f'{path.name}: locked-test blind artifact tidak boleh memuat y_silver.')
    validate_probability_matrix(frame[model_probability_columns(prefix)].to_numpy(), path.name)


def fold_outputs_complete(plan: FoldPlan, fold_dir: Path) -> bool:
    if not all(path.exists() for path in required_fold_paths(fold_dir)):
        return False
    try:
        if not model_export_ready(fold_dir / 'models' / 'indobert_outer_refit_model'):
            raise ValueError('IndoBERT outer-train refit export tidak lengkap.')
        prediction_dir = fold_dir / 'predictions'
        validate_prediction_file(prediction_dir / 'svm_oof.csv', 'svm', TRAIN_ROW_IDS[plan.outer_val_idx], True)
        validate_prediction_file(prediction_dir / 'bert_oof.csv', 'bert', TRAIN_ROW_IDS[plan.outer_val_idx], True)
        validate_prediction_file(prediction_dir / 'svm_validation.csv', 'svm', VALIDATION_ROW_IDS, True)
        validate_prediction_file(prediction_dir / 'bert_validation.csv', 'bert', VALIDATION_ROW_IDS, True)
        validate_prediction_file(prediction_dir / 'svm_locked_test_blind.csv', 'svm', LOCKED_TEST_ROW_IDS, False)
        validate_prediction_file(prediction_dir / 'bert_locked_test_blind.csv', 'bert', LOCKED_TEST_ROW_IDS, False)
        return True
    except Exception as error:
        print(f'Fold {plan.outer_fold}: output lama tidak valid, akan diproses ulang/resume. Detail: {error}')
        return False


def run_one_fold(seed_dir: Path, plan: FoldPlan) -> None:
    fold_dir = fold_directory(seed_dir, plan.outer_fold)
    if fold_outputs_complete(plan, fold_dir):
        mark_fold_state(seed_dir, plan.outer_fold, 'completed', reused_at_utc=utc_now())
        print(f'Fold {plan.outer_fold}: completed artifact valid, dilewati.')
        return

    mark_fold_state(seed_dir, plan.outer_fold, 'running', started_at_utc=utc_now())
    print(f'\n===== SEED FOLD {plan.outer_fold}/{OUTER_N_SPLITS} =====')
    try:
        svm_result = run_svm_fold(plan, fold_dir)
        bert_model, bert_tokenizer, bert_training = train_or_load_bert_fold(plan, seed_dir, fold_dir)
        prediction_dir = fold_dir / 'predictions'

        bert_oof_texts = train_df.iloc[plan.outer_val_idx][BERT_CONFIG['text_column']].fillna('').astype(str).tolist()
        bert_validation_texts = validation_df[BERT_CONFIG['text_column']].fillna('').astype(str).tolist()
        bert_oof_proba = predict_probabilities_torch(bert_model, bert_oof_texts, bert_tokenizer)
        bert_validation_proba = predict_probabilities_torch(bert_model, bert_validation_texts, bert_tokenizer)
        bert_locked_test_proba = predict_probabilities_torch(bert_model, LOCKED_TEST_BERT_TEXTS, bert_tokenizer)

        atomic_write_csv(
            prediction_dir / 'bert_oof.csv',
            prediction_frame(TRAIN_ROW_IDS[plan.outer_val_idx], bert_oof_proba, 'bert', plan.outer_fold, train_y[plan.outer_val_idx]),
        )
        atomic_write_csv(
            prediction_dir / 'bert_validation.csv',
            prediction_frame(VALIDATION_ROW_IDS, bert_validation_proba, 'bert', y_silver=validation_y),
        )
        atomic_write_csv(
            prediction_dir / 'bert_locked_test_blind.csv',
            prediction_frame(LOCKED_TEST_ROW_IDS, bert_locked_test_proba, 'bert'),
        )

        bert_result = {
            'model': 'IndoBERT', 'fold': plan.outer_fold,
            'oof_metrics': metric_summary(train_y[plan.outer_val_idx], bert_oof_proba),
            'training': bert_training,
        }
        summary = {
            'seed': int(plan.bert_seed // 10_000), 'outer_fold': plan.outer_fold,
            'svm': svm_result, 'indobert': bert_result,
            'completed_at_utc': utc_now(),
        }
        atomic_write_json(fold_dir / 'fold_summary.json', summary)

        if not fold_outputs_complete(plan, fold_dir):
            raise RuntimeError('Fold menghasilkan output yang tidak lolos validasi akhir.')
        checksums = {str(path.relative_to(fold_dir)): sha256_file(path) for path in required_fold_paths(fold_dir)}
        mark_fold_state(seed_dir, plan.outer_fold, 'completed', completed_at_utc=utc_now(), output_checksums=checksums)
        print('Fold {}: completed. SVM OOF F1={:.4f}; BERT OOF F1={:.4f}'.format(
            plan.outer_fold, svm_result['oof_metrics']['macro_f1'], bert_result['oof_metrics']['macro_f1']
        ))
    except Exception as error:
        mark_fold_state(seed_dir, plan.outer_fold, 'failed', failed_at_utc=utc_now(), error=repr(error), traceback=traceback.format_exc())
        raise
    finally:
        if 'bert_model' in locals():
            del bert_model
        if 'bert_tokenizer' in locals():
            del bert_tokenizer
        clear_accelerator_memory()


### Aggregate fold probabilities into per-seed base-model outputs

Combines held-out OOF predictions and five-fold validation and locked-test ensembles, derives the twelve stacking meta-features, and calculates private base-model metric summaries for train-derived partitions.


In [ ]:
def load_fold_prediction(fold_dir: Path, filename: str) -> pd.DataFrame:
    path = fold_dir / 'predictions' / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def aggregate_oof(seed_dir: Path, prefix: str, plans: list[FoldPlan]) -> pd.DataFrame:
    frames = [load_fold_prediction(fold_directory(seed_dir, plan.outer_fold), f'{prefix}_oof.csv') for plan in plans]
    combined = pd.concat(frames, ignore_index=True).sort_values('row_id').reset_index(drop=True)
    if not np.array_equal(combined['row_id'].to_numpy(dtype=np.int64), TRAIN_ROW_IDS):
        raise RuntimeError(f'{prefix} OOF row coverage/order tidak lengkap.')
    if combined['row_id'].duplicated().any():
        raise RuntimeError(f'{prefix} OOF memiliki row duplikat.')
    if not np.array_equal(combined['y_silver'].to_numpy(dtype=np.int64), train_y):
        raise RuntimeError(f'{prefix} OOF y_silver tidak cocok dengan train frozen.')
    validate_probability_matrix(combined[model_probability_columns(prefix)].to_numpy(), f'{prefix} OOF aggregate')
    return combined


def aggregate_ensemble(seed_dir: Path, prefix: str, plans: list[FoldPlan], split_name: str, row_ids: np.ndarray, y_silver: np.ndarray | None) -> pd.DataFrame:
    suffix = 'validation.csv' if split_name == 'validation' else 'locked_test_blind.csv'
    frames = [load_fold_prediction(fold_directory(seed_dir, plan.outer_fold), f'{prefix}_{suffix}') for plan in plans]
    probabilities = []
    expected_ids = row_ids.astype(np.int64)
    for frame in frames:
        if not np.array_equal(frame['row_id'].to_numpy(dtype=np.int64), expected_ids):
            raise RuntimeError(f'{prefix} {split_name}: row id tidak konsisten antar fold.')
        if split_name == 'locked_test' and 'y_silver' in frame.columns:
            raise RuntimeError('Locked-test fold artifact tidak lagi blind.')
        probabilities.append(frame[model_probability_columns(prefix)].to_numpy())
    ensemble = np.mean(np.stack(probabilities, axis=0), axis=0)
    validate_probability_matrix(ensemble, f'{prefix} {split_name} ensemble')
    return prediction_frame(expected_ids, ensemble, prefix, y_silver=y_silver)


def combine_base_models(svm_frame: pd.DataFrame, bert_frame: pd.DataFrame, include_outer_fold: bool, include_y: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    join_columns = ['row_id'] + (['outer_fold'] if include_outer_fold else []) + (['y_silver'] if include_y else [])
    svm_keep = join_columns + [column for column in svm_frame.columns if column.startswith('svm_')]
    bert_keep = join_columns + [column for column in bert_frame.columns if column.startswith('bert_')]
    merged = svm_frame[svm_keep].merge(bert_frame[bert_keep], on=join_columns, validate='one_to_one')

    svm_proba = merged[model_probability_columns('svm')].to_numpy()
    bert_proba = merged[model_probability_columns('bert')].to_numpy()
    merged['pred_disagreement'] = (merged['svm_prediction'] != merged['bert_prediction']).astype(np.int64)
    merged['prob_disagreement'] = 0.5 * np.abs(svm_proba - bert_proba).sum(axis=1)

    metadata_columns = join_columns
    meta_features = merged[metadata_columns + META_FEATURE_NAMES].copy()
    return merged, meta_features


def file_manifest(seed_dir: Path, paths: list[Path]) -> dict[str, str]:
    return {str(path.relative_to(seed_dir)): sha256_file(path) for path in paths if path.exists()}


def aggregate_seed(seed: int, seed_dir: Path, plans: list[FoldPlan]) -> pd.DataFrame:
    incomplete = [plan.outer_fold for plan in plans if not fold_outputs_complete(plan, fold_directory(seed_dir, plan.outer_fold))]
    if incomplete:
        raise RuntimeError(f'Seed {seed} belum lengkap. Fold belum valid: {incomplete}')

    oof_svm = aggregate_oof(seed_dir, 'svm', plans)
    oof_bert = aggregate_oof(seed_dir, 'bert', plans)
    val_svm = aggregate_ensemble(seed_dir, 'svm', plans, 'validation', VALIDATION_ROW_IDS, validation_y)
    val_bert = aggregate_ensemble(seed_dir, 'bert', plans, 'validation', VALIDATION_ROW_IDS, validation_y)
    test_svm = aggregate_ensemble(seed_dir, 'svm', plans, 'locked_test', LOCKED_TEST_ROW_IDS, None)
    test_bert = aggregate_ensemble(seed_dir, 'bert', plans, 'locked_test', LOCKED_TEST_ROW_IDS, None)

    oof_base, oof_meta = combine_base_models(oof_svm, oof_bert, include_outer_fold=True, include_y=True)
    val_base, val_meta = combine_base_models(val_svm, val_bert, include_outer_fold=False, include_y=True)
    test_base, test_meta = combine_base_models(test_svm, test_bert, include_outer_fold=False, include_y=False)

    output_paths = {
        'oof_base': seed_dir / 'oof_train_base_probabilities.csv',
        'validation_base': seed_dir / 'validation_ensemble_base_probabilities.csv',
        'locked_test_base': seed_dir / 'locked_test_ensemble_base_probabilities_blind.csv',
        'oof_meta': seed_dir / 'oof_train_meta_features.csv',
        'validation_meta': seed_dir / 'validation_meta_features.csv',
        'locked_test_meta': seed_dir / 'locked_test_meta_features_blind.csv',
    }
    atomic_write_csv(output_paths['oof_base'], oof_base)
    atomic_write_csv(output_paths['validation_base'], val_base)
    atomic_write_csv(output_paths['locked_test_base'], test_base)
    atomic_write_csv(output_paths['oof_meta'], oof_meta)
    atomic_write_csv(output_paths['validation_meta'], val_meta)
    atomic_write_csv(output_paths['locked_test_meta'], test_meta)

    metrics_rows = []
    for split_name, y_true, svm_frame, bert_frame in [
        ('oof_train', train_y, oof_svm, oof_bert),
        ('validation', validation_y, val_svm, val_bert),
    ]:
        for model_name, frame, prefix in [('SVM', svm_frame, 'svm'), ('IndoBERT', bert_frame, 'bert')]:
            metrics_rows.append({'seed': seed, 'split': split_name, 'model': model_name, **metric_summary(y_true, frame[model_probability_columns(prefix)].to_numpy())})
    metrics_frame = pd.DataFrame(metrics_rows)
    atomic_write_csv(seed_dir / 'base_model_oof_validation_metrics.csv', metrics_frame)

    schema = {
        'label_order': LABEL_NAMES, 'meta_feature_names': META_FEATURE_NAMES,
        'entropy': 'normalized Shannon entropy with natural logarithm divided by log(3)',
        'pred_disagreement': '1 when SVM and IndoBERT argmax class differ, otherwise 0',
        'prob_disagreement': 'total variation distance = 0.5 * sum(abs(svm_probability - bert_probability))',
    }
    atomic_write_json(seed_dir / 'schema.json', schema)

    output_file_paths = list(output_paths.values()) + [seed_dir / 'base_model_oof_validation_metrics.csv', seed_dir / 'schema.json']
    run_manifest = {
        'seed': seed, 'protocol_hash': PROTOCOL_HASH, 'input_hashes': input_hashes,
        'completed_at_utc': utc_now(), 'locked_test_metrics_computed': False,
        'locked_test_artifact_is_label_blind': True,
        'fold_count': OUTER_N_SPLITS, 'meta_feature_names': META_FEATURE_NAMES,
        'output_checksums': file_manifest(seed_dir, output_file_paths),
        'environment': {
            'python': sys.version, 'torch': torch.__version__, 'transformers': transformers.__version__,
            'device': str(DEVICE), 'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
        },
    }
    atomic_write_json(seed_dir / 'run_manifest.json', run_manifest)

    state = read_json(seed_state_path(seed_dir))
    state['status'] = 'seed_completed'
    state['completed_at_utc'] = utc_now()
    state['run_manifest'] = str((seed_dir / 'run_manifest.json').relative_to(ARTIFACT_ROOT))
    update_seed_state(seed_dir, state)

    print(f'\nSeed {seed} completed. Base metrics (OOF/validation only):')
    display(metrics_frame.round(4))
    print('Locked-test labels and metrics were not read or computed in R1_06.')
    return metrics_frame


### Execute requested seeds with resumability

Runs or resumes each requested seed, skips completed compatible artifacts, aggregates its fold outputs, and writes the authorized per-seed execution records. The per-row predictions, fold models, and run-state files are not released.


In [ ]:
if not set(RUN_SEEDS).issubset(PLANNED_SEEDS):
    raise ValueError(f'RUN_SEEDS harus subset dari PLANNED_SEEDS={PLANNED_SEEDS}.')

all_metric_frames = []
for seed in RUN_SEEDS:
    print(f'\n' + '=' * 88)
    print(f'R1_06 BASE-MODEL PIPELINE | SEED {seed}')
    print('=' * 88)
    seed_dir, state = load_or_create_seed_state(seed)
    plans = make_seed_plan(seed)
    if not (seed_dir / 'fold_structure.json').exists():
        persist_seed_plan(seed_dir, plans)

    new_folds_completed = 0
    for plan in plans:
        fold_dir = fold_directory(seed_dir, plan.outer_fold)
        already_complete = fold_outputs_complete(plan, fold_dir)
        if not already_complete:
            run_one_fold(seed_dir, plan)
            new_folds_completed += 1
            if MAX_NEW_FOLDS_PER_RUN is not None and new_folds_completed >= MAX_NEW_FOLDS_PER_RUN:
                print(f'Safe stop reached after {new_folds_completed} new fold(s). Run All again to continue this seed.')
                break
        else:
            print(f'Fold {plan.outer_fold}: sudah lengkap, skip.')

    remaining = [plan.outer_fold for plan in plans if not fold_outputs_complete(plan, fold_directory(seed_dir, plan.outer_fold))]
    if remaining:
        print(f'Seed {seed} belum selesai. Fold tersisa: {remaining}. Tidak ada ensemble atau metric akhir yang dibuat.')
        continue

    state = read_json(seed_state_path(seed_dir))
    if state.get('status') == 'seed_completed' and (seed_dir / 'run_manifest.json').exists():
        print(f'Seed {seed}: artifact aggregate sudah completed, tidak dibangun ulang.')
        all_metric_frames.append(pd.read_csv(seed_dir / 'base_model_oof_validation_metrics.csv'))
    else:
        all_metric_frames.append(aggregate_seed(seed, seed_dir, plans))

if all_metric_frames:
    selected_metrics = pd.concat(all_metric_frames, ignore_index=True)
    print('\nR1_06 selesai untuk seed lengkap yang dipilih. Tidak ada metric locked test di bawah ini.')
    display(selected_metrics.round(4))
else:
    print('Tidak ada seed lengkap pada eksekusi ini. Periksa status fold lalu jalankan kembali notebook dari awal.')
